# buffer-copy_-inplace — faded example 1: Complete the in-place EMA buffer update with copy_

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `buffer-copy_-inplace`. Running the beacon reports progress on the `PyTorch: in-place buffer copy` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: in-place buffer copy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`buffer-copy_-inplace`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "buffer-copy_-inplace"
DD_SUBTOPIC = "PyTorch: in-place buffer copy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To update a registered buffer you overwrite its storage with `buffer.copy_(new_value)`, which preserves `id(buffer)`. The new value is the EMA `(1 - momentum) * buffer + momentum * batch`. Computing the value in a fresh tensor and then `copy_`-ing it in is the canonical safe route.

## Faded exercise 1

Implement `update_running_mean(running_mean, batch_mean, momentum)` so it updates `running_mean` IN PLACE to `(1 - momentum) * running_mean + momentum * batch_mean`. The EMA value is already computed for you; you must write the single line that copies it into the buffer's existing storage so `id(running_mean)` is preserved.

**Fill in:** Writes the computed EMA value into running_mean's existing storage in place via copy_.

In [ ]:
def update_running_mean(running_mean: Tensor, batch_mean: Tensor, momentum: float) -> None:
    new_value = (1 - momentum) * running_mean + momentum * batch_mean
    running_mean.copy_(new_value)


def _test():
    t.manual_seed(0)
    rm = t.tensor([1.0, 2.0, 3.0, 4.0])
    bm = t.tensor([5.0, 6.0, 7.0, 8.0])
    m = 0.1
    expected = (1 - m) * rm + m * bm
    before_id = id(rm)
    ret = update_running_mean(rm, bm, m)
    assert ret is None, "function must return None (mutate in place)"
    assert id(rm) == before_id, "buffer identity must be preserved"
    assert t.allclose(rm, expected), f"got {rm}, expected {expected}"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def update_running_mean(running_mean: Tensor, batch_mean: Tensor, momentum: float) -> None:
    new_value = (1 - momentum) * running_mean + momentum * batch_mean
    running_mean.copy_(new_value)
```
</details>